# Functions

These are the pieces reused by the experiment notebooks. Run this notebook with `%run ./functions.ipynb`; it defines functions without starting an experiment.

The names and alternating updates follow my original notebooks. Duplicate definitions and unused imports have been removed.

In [ ]:
import hashlib
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.optimize import linear_sum_assignment
from sklearn import metrics
from sklearn.cluster import KMeans, kmeans_plusplus
from sklearn.datasets import load_digits
from sklearn.decomposition import NMF
from sklearn.mixture import GaussianMixture

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Data and scores

MNIST and Fashion-MNIST use train + test together by default, as in the thesis clustering experiments. Set `split="train"` or `"test"` for separate evaluation. `digits` is a small offline example, not a substitute for MNIST results. Subsets get a new consecutive row order, so each row of $Y$ stays attached to the right sample.

ACC uses a one-to-one Hungarian match. Unmatched clusters remain unmatched; they are not assigned the most common true label. F1 is weighted F1 after this match. Labels are only used in these reporting functions.

In [ ]:
def load_data(name='mnist', n_samples=None, seed=42, data_dir='data', split='all'):
    name = name.lower().replace('-', '_')
    if split not in ('all', 'train', 'test'):
        raise ValueError("split must be 'all', 'train', or 'test'")
    if name == 'digits':
        if split != 'all':
            raise ValueError("The small sklearn digits example only has split='all'.")
        data = load_digits()
        X, y = data.data.astype(np.float32) / 16., data.target
    elif name in ('mnist', 'fashion_mnist'):
        from torchvision.datasets import MNIST, FashionMNIST
        cls = MNIST if name == 'mnist' else FashionMNIST
        parts = [True, False] if split == 'all' else [split == 'train']
        datasets = [cls(data_dir, train=train, download=True) for train in parts]
        X = torch.cat([d.data for d in datasets]).flatten(1).float() / 255.
        y = torch.cat([d.targets for d in datasets]).numpy()
    elif name == 'reuters10k':
        if split != 'all':
            raise ValueError("Reuters NPZ is the prepared clustering set; use split='all'.")
        path = Path(data_dir) / 'reuters10k.npz'
        if not path.exists():
            raise FileNotFoundError(f'{path} is missing. Prepare it in reuters_fashion.ipynb first.')
        with np.load(path, allow_pickle=False) as data:
            if {'data', 'indices', 'indptr', 'shape'} <= set(data.files):
                X = sparse.csr_matrix((data['data'], data['indices'], data['indptr']),
                                      shape=tuple(data['shape']))
            else:
                X = data['X'] if 'X' in data else data['data']
            key = next((k for k in ('y', 'labels', 'label') if k in data), None)
            if key is None:
                raise ValueError('Reuters NPZ must contain y, labels, or label.')
            y = data[key]
    else:
        raise ValueError('Use mnist, fashion_mnist, reuters10k, or digits.')
    y = np.asarray(y).reshape(-1)
    if X.shape[0] != len(y):
        raise ValueError('X and y have different numbers of samples.')
    if n_samples is not None:
        if n_samples < 1:
            raise ValueError('n_samples must be positive or None.')
        indices = np.random.default_rng(seed).permutation(len(y))[:n_samples]
        X, y = X[indices], y[indices]
    if sparse.issparse(X):
        # Reuters-10k x 2,000 occupies about 80 MB after this conversion.
        X = X.toarray()
    X = torch.as_tensor(X, dtype=torch.float32).cpu().contiguous()
    if X.ndim != 2 or not torch.isfinite(X).all():
        raise ValueError('Expected a finite [samples, features] matrix.')
    return X, y.astype(np.int64)


def aligned_labels(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    true_ids, cluster_ids = np.unique(y), np.unique(pred)
    table = metrics.cluster.contingency_matrix(y, pred)
    rows, cols = linear_sum_assignment(-table)
    unmatched = min(int(true_ids.min()), -1) - 1
    mapping = {cluster_ids[c]: true_ids[r] for r, c in zip(rows, cols)}
    return np.array([mapping.get(p, unmatched) for p in pred])


def evaluate(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    if y.ndim != 1 or pred.ndim != 1 or len(y) != len(pred) or not len(y):
        raise ValueError('y and pred must be nonempty label vectors of equal length.')
    matched = aligned_labels(y, pred)
    table = metrics.cluster.contingency_matrix(y, pred)
    return (float(np.mean(y == matched)),
            metrics.normalized_mutual_info_score(y, pred, average_method='arithmetic'),
            metrics.adjusted_rand_score(y, pred),
            metrics.f1_score(y, matched, labels=np.unique(y), average='weighted', zero_division=0),
            float(table.max(axis=0).sum() / table.sum()))


def score_dict(y, pred):
    return dict(zip(('ACC', 'NMI', 'ARI', 'F1', 'Purity'), evaluate(y, pred)))


def kmeans_fun_gpu(X, K=10, max_iter=100, batch_size=4096, tol=1e-5):
    X = torch.as_tensor(X, dtype=torch.float32, device=device)
    if X.ndim != 2 or not 1 <= K <= len(X) or max_iter < 1:
        raise ValueError('Need a 2D matrix, 1 <= K <= samples, and max_iter >= 1.')
    centers = X[torch.randperm(len(X), device=device)[:K]].clone()
    with torch.no_grad():
        for _ in range(max_iter):
            labels = torch.cat([torch.cdist(batch, centers).argmin(1)
                                for batch in X.split(batch_size)])
            previous = centers.clone()
            for k in range(K):
                points = X[labels == k]
                centers[k] = points.mean(0) if len(points) else X[torch.randint(len(X), ())]
            if torch.linalg.vector_norm(centers - previous, dim=1).sum() <= tol:
                break
        labels = torch.cat([torch.cdist(batch, centers).argmin(1)
                            for batch in X.split(batch_size)])
    return centers.cpu().numpy(), labels.cpu().numpy()

## AE and VAE

The default dense architecture is supplied by each experiment. For the thesis setting use `[D, 500, 500, 2000, 10]`. Clustering uses the posterior mean $\mu$; the decoder sees $z=\mu+\epsilon\exp(\tfrac12\log\sigma^2)$. Dropout is only used in the encoder.

The convolutional model is reconstructed from Figure 5: two stride-2 convolutions, two residual blocks, and mean/log-variance heads. It accepts 28×28 images. The original Reuters reshape rule was not retained, so text is not arbitrarily reshaped into an image here. For a linear decoder output in a dense model, pass `final_activation=nn.Identity()`.

In [ ]:
def build_units(dimensions, dropout=0.):
    layers = []
    for left, right in zip(dimensions[:-1], dimensions[1:]):
        layers += [nn.Linear(left, right), nn.ReLU()]
        if dropout:
            layers.append(nn.Dropout(dropout))
    return layers


def initialise_weights(module):
    if isinstance(module, (nn.Linear, nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.xavier_uniform_(module.weight, gain=nn.init.calculate_gain('relu'))
        if module.bias is not None:
            nn.init.zeros_(module.bias)


class AutoEncoder(nn.Module):
    def __init__(self, dimensions, dropout=0.2, final_activation=None):
        super().__init__()
        self.dimensions = list(dimensions)
        self.hidden_dimension = dimensions[-1]
        self.encoder = nn.Sequential(*build_units(dimensions[:-1], dropout),
                                     nn.Linear(dimensions[-2], dimensions[-1]))
        reverse = list(reversed(dimensions))
        self.decoder = nn.Sequential(*build_units(reverse[:-1]),
                                     nn.Linear(reverse[-2], reverse[-1]),
                                     nn.Sigmoid() if final_activation is None else final_activation)
        self.apply(initialise_weights)

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        return self.decoder(z), z


class VariationalAutoEncoder(nn.Module):
    def __init__(self, dimensions, dropout=0.2, final_activation=None):
        super().__init__()
        self.dimensions = list(dimensions)
        self.hidden_dimension = dimensions[-1]
        self.encoder = nn.Sequential(*build_units(dimensions[:-1], dropout),
                                     nn.Linear(dimensions[-2], 2 * dimensions[-1]))
        reverse = list(reversed(dimensions))
        self.decoder = nn.Sequential(*build_units(reverse[:-1]),
                                     nn.Linear(reverse[-2], reverse[-1]),
                                     nn.Sigmoid() if final_activation is None else final_activation)
        self.apply(initialise_weights)

    def encode_stats(self, x):
        mu, log_var = self.encoder(x).chunk(2, dim=1)
        return mu, log_var.clamp(-20, 10)

    def encode(self, x):
        return self.encode_stats(x)[0]

    @staticmethod
    def reparameterize(mu, log_var):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * log_var)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode_stats(x)
        z = self.reparameterize(mu, log_var)
        return self.decoder(z), mu, log_var


class ResidualBlock(nn.Module):
    def __init__(self, channels=64, dropout=0.):
        super().__init__()
        self.layers = nn.Sequential(nn.ReLU(), nn.Conv2d(channels, channels, 3, padding=1),
                                    nn.ReLU(), nn.Dropout2d(dropout),
                                    nn.Conv2d(channels, channels, 3, padding=1),
                                    nn.ReLU(), nn.Dropout2d(dropout),
                                    nn.Conv2d(channels, channels, 1))

    def forward(self, x):
        return x + self.layers(x)


class ConvolutionalVAE(nn.Module):
    def __init__(self, input_dim=784, latent_dim=10, dropout=0.2):
        super().__init__()
        if input_dim != 784:
            raise ValueError('This convolutional model expects flattened 28 x 28 images.')
        self.hidden_dimension = latent_dim
        self.encoder = nn.Sequential(nn.Unflatten(1, (1, 28, 28)),
                                     nn.Conv2d(1, 32, 4, 2, 1), nn.ReLU(), nn.Dropout2d(dropout),
                                     nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(), nn.Dropout2d(dropout),
                                     ResidualBlock(64, dropout), ResidualBlock(64, dropout),
                                     nn.Flatten(), nn.Linear(64 * 7 * 7, 2 * latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 64 * 7 * 7),
                                     nn.Unflatten(1, (64, 7, 7)),
                                     ResidualBlock(64), ResidualBlock(64), nn.ReLU(),
                                     nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
                                     nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Sigmoid(), nn.Flatten())
        self.apply(initialise_weights)

    def encode_stats(self, x):
        mu, log_var = self.encoder(x).chunk(2, dim=1)
        return mu, log_var.clamp(-20, 10)

    def encode(self, x):
        return self.encode_stats(x)[0]

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode_stats(x)
        z = VariationalAutoEncoder.reparameterize(mu, log_var)
        return self.decoder(z), mu, log_var

## Loss and pretraining

Reconstruction sums over input features and averages over samples; KL sums over latent coordinates and averages over samples. `loss="bce"` follows the original normalized-image code. `loss="mse"` supports the Gaussian reconstruction formulation and TF-IDF experiments.

`pre_train` uses a fixed `beta` unless `beta_start` is supplied, in which case it warms up over `warm_epochs`. AE models return `(recon, z)` and have zero KL. Log variance is bounded for numerical stability, and gradients are clipped at norm 10.

In [ ]:
def Loss_Function(recon, batch, mu, log_var, kind='bce'):
    if kind == 'bce':
        reconstruction = F.binary_cross_entropy(recon, batch, reduction='none').flatten(1).sum(1).mean()
    elif kind == 'mse':
        reconstruction = (recon - batch).square().flatten(1).sum(1).mean()
    else:
        raise ValueError("loss must be 'bce' or 'mse'.")
    kl = -0.5 * (1 + log_var - mu.square() - log_var.exp()).sum(1).mean()
    return reconstruction, kl


def model_loss(model, x, kind):
    output = model(x)
    if len(output) == 3:
        recon, mu, log_var = output
        rec, kl = Loss_Function(recon, x, mu, log_var, kind)
    else:
        recon, mu = output
        rec, _ = Loss_Function(recon, x, torch.zeros_like(mu), torch.zeros_like(mu), kind)
        kl = x.new_zeros(())
    return rec, kl, mu


@torch.no_grad()
def vae_encode(X, model, batch_size=256):
    was_training = model.training
    model.eval()
    model_device = next(model.parameters()).device
    X = torch.as_tensor(X, dtype=torch.float32)
    Z = torch.cat([model.encode(x.to(model_device)).cpu() for x in X.split(batch_size)])
    model.train(was_training)
    return Z.numpy()


def optimizer_step(value, optimizer):
    if not torch.isfinite(value):
        raise FloatingPointError('Nonfinite loss: check inputs and learning rates.')
    optimizer.zero_grad(set_to_none=True)
    value.backward()
    parameters = [p for group in optimizer.param_groups for p in group['params']]
    nn.utils.clip_grad_norm_(parameters, max_norm=10., error_if_nonfinite=True)
    optimizer.step()


def pre_train(X, model, epochs=10, batch_size=256, lr=1e-3, beta=1., loss='bce',
              seed=42, beta_start=None, warm_epochs=50):
    seed_everything(seed)
    X = torch.as_tensor(X, dtype=torch.float32).cpu()
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {key: [] for key in ('loss', 'recon_loss', 'kl_loss', 'beta')}
    for epoch in range(epochs):
        model.train()
        weight = beta if beta_start is None else beta_start + (beta - beta_start) * min(epoch / max(warm_epochs, 1), 1.)
        order = torch.randperm(len(X), generator=torch.Generator().manual_seed(seed + epoch))
        total = np.zeros(3)
        for idx in order.split(batch_size):
            x = X[idx].to(device)
            rec, kl, _ = model_loss(model, x, loss)
            value = rec + weight * kl
            optimizer_step(value, optimizer)
            total += np.array([value.item(), rec.item(), kl.item()]) * len(x)
        for key, value in zip(('loss', 'recon_loss', 'kl_loss'), total / len(X)):
            history[key].append(float(value))
        history['beta'].append(float(weight))
    return history

## Indicator updates

`prox_primp` solves the scalar PRIMP step on $[0,1]$, including strengths large enough for the printed piecewise intervals to overlap. `improved_primp` keeps the thesis update: `root = mean(top two values)` and $Y\leftarrow\operatorname{clip}(Y+\kappa(Y-\mathrm{root}),0,1)$. This adaptive update is a heuristic; ties and zero rows do not guarantee a one-hot result.

`thesis_exclusive_shrink` retains Eq. 65. It is **not** the exact proximal map of the squared row $\ell_1$ norm in Eq. 64. `prox_exclusive_lasso` implements the exact sorted-threshold solution as a separate option. Applying exclusive shrinkage followed by the binary step is operator splitting, not a proven proximal map of their sum. `kappa` and `gamma` are step strengths here; they are not multiplied by Adam’s learning rate again.

The `legacy` and `legacy_alt` modes preserve the early accumulated-multiplier experiments, with separate `alphaX` and `prox_lr`. Diagnostics include GDR (zero row → 0), near-zero entries, and actual one-hot rows.

In [ ]:
def prox_primp(v, kappa):
    if kappa < 0:
        raise ValueError('kappa must be nonnegative.')
    low = (v - 2 * kappa).clamp(0, 0.5)
    high = (v + 2 * kappa).clamp(0.5, 1)
    low_value = 0.5 * (low - v).square() + kappa * (1 - (1 - 2 * low).abs())
    high_value = 0.5 * (high - v).square() + kappa * (1 - (1 - 2 * high).abs())
    return torch.where(low_value <= high_value, low, high)


def improved_primp(v, kappa):
    if kappa < 0 or v.shape[1] < 2:
        raise ValueError('Need kappa >= 0 and at least two clusters.')
    root = v.topk(2, dim=1).values.mean(1, keepdim=True)
    return (v + kappa * (v - root)).clamp(0, 1)


def prox_exclusive_lasso(v, gamma):
    if gamma < 0:
        raise ValueError('gamma must be nonnegative.')
    ordered = v.abs().sort(dim=1, descending=True).values
    count = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    thresholds = 2 * gamma * ordered.cumsum(1) / (1 + 2 * gamma * count)
    threshold = thresholds.max(dim=1, keepdim=True).values
    return v.sign() * (v.abs() - threshold).clamp_min(0)


def thesis_exclusive_shrink(v, gamma):
    if gamma < 0:
        raise ValueError('gamma must be nonnegative.')
    return v.sign() * (v.abs() - 2 * gamma * v.abs().sum(1, keepdim=True)).clamp_min(0)


@torch.no_grad()
def assignment_diagnostics(Y, tol=1e-6):
    Y = torch.as_tensor(Y)
    absolute = Y.abs()
    row_sum = absolute.sum(1)
    gdr = torch.where(row_sum > 0, absolute.max(1).values / row_sum.clamp_min(1e-12), 0.)
    binary = ((Y.abs() < tol) | ((Y - 1).abs() < tol)).all(1)
    one_hot = binary & ((Y - 1).abs() < tol).sum(1).eq(1)
    return {'GDR': float(gdr.mean()), 'zero_count': int((absolute < tol).sum()),
            'zero_fraction': float((absolute < tol).float().mean()),
            'one_hot_fraction': float(one_hot.float().mean()),
            'binary_penalty': float((1 - (1 - 2 * Y.clamp(0, 1)).abs()).mean()),
            'exclusive_penalty': float(row_sum.square().mean())}


class MatrixFactorizationBinaryModel(nn.Module):
    def __init__(self, n_samples, n_clusters, latent_dim, Y_init=None, C_init=None):
        super().__init__()
        self.Y = nn.Parameter(torch.rand(n_samples, n_clusters))
        self.C = nn.Parameter(torch.rand(n_clusters, latent_dim))
        self.register_buffer('lambdasX', torch.zeros_like(self.Y))
        if Y_init is not None:
            self.Y.data.copy_(torch.as_tensor(Y_init, dtype=torch.float32))
        if C_init is not None:
            self.C.data.copy_(torch.as_tensor(C_init, dtype=torch.float32))

    def forward(self, indices=None):
        Y = self.Y if indices is None else self.Y[indices]
        return Y @ self.C

    @torch.no_grad()
    def apply_constraints(self, indices, kappa, gamma, binary_mode, exclusive_mode,
                          bounded=True, alphaX=1e-8, prox_lr=1e-4):
        Y = self.Y[indices].clamp_min(0)
        if bounded or binary_mode != 'none':
            Y = Y.clamp_max(1)
        if exclusive_mode == 'thesis':
            Y = thesis_exclusive_shrink(Y, gamma)
        elif exclusive_mode == 'exact':
            Y = prox_exclusive_lasso(Y, gamma)
        elif exclusive_mode != 'none':
            raise ValueError("exclusive_mode must be 'thesis', 'exact', or 'none'.")
        if binary_mode == 'improved':
            Y = improved_primp(Y, kappa)
        elif binary_mode == 'primp':
            Y = prox_primp(Y, kappa)
        elif binary_mode == 'legacy':
            # The accumulated multiplier version used in the early notebooks.
            root = Y.topk(2, dim=1).values.mean(1, keepdim=True)
            self.lambdasX[indices] += alphaX * (Y - root)
            Y = (Y + prox_lr * self.lambdasX[indices]).clamp(0, 1)
        elif binary_mode == 'legacy_alt':
            L = self.lambdasX[indices]
            Y = (Y + 2 * prox_lr * L * (2 * (Y > .5).float() - 1)).clamp(0, 1)
            self.lambdasX[indices] += alphaX * ((1 - (1 - 2 * Y).abs()) - 1)
        elif binary_mode != 'none':
            raise ValueError("binary_mode must be 'improved', 'primp', 'legacy', 'legacy_alt', or 'none'.")
        self.Y[indices] = Y

## Joint VAE + Semi-NMF

The smooth update uses

$$L=L_{\mathrm{recon}}+\beta L_{\mathrm{KL}}+\lambda\,\operatorname{mean}((\mu-YC)^2).$$

The factorization term uses elementwise MSE, matching the uploaded code. `loss` in the history is this smooth loss averaged over the C and Y steps; binary and exclusive penalties are separate diagnostics, not terms counted twice in backpropagation.

Two Adam optimizers retain separate states for the shared network: first `(C, VAE)` with Y detached, then `(Y, VAE)` with C detached and a fresh forward pass. Only the sampled Y rows are changed; the original dense Adam could otherwise move unsampled rows through momentum. Y is bounded in `[0,1]`; C remains signed.

Beta continues its warm-up from pretraining; the MF weight starts its own warm-up from 0.1 toward 1 at the first joint epoch. Set `restart_kl=True` to repeat the KL warm-up as in the older experiment. Kappa/gamma double every 50 joint epochs. The short notebook runs are not the thesis training budget. `checkpoint` saves the latest epoch, both optimizer states and sample-order fingerprint; `resume` continues to the requested total `epochs`, without selecting by labels. Optional cyclic MF weights, delayed indicator updates and periodic center reinitialization retain the older experiments; they are off by default.

In [ ]:
def train_vae_mf_model_optimized(X, n_clusters=10, dimensions=None, model=None,
        pretrain_epochs=10, epochs=50, batch_size=256, lr=1e-3,
        beta_start=.1, beta_end=1., warm_epochs=50, mf_weight=1., kappa=1e-4,
        gamma=1e-3, binary_mode='improved', exclusive_mode='thesis', double_every=50,
        loss='bce', seed=42, y=None, verbose=True, checkpoint=None, resume=None,
        lr_Y=None, lr_C=None, scheduler_step_size=50, scheduler_rate=.5,
        mf_cycle=None, prox_start_epoch=0, reinitialize_every=None, alphaX=1e-8, prox_lr=1e-4,
        restart_kl=False):
    seed_everything(seed)
    X = torch.as_tensor(X, dtype=torch.float32).cpu().contiguous()
    if X.ndim != 2 or not 2 <= n_clusters <= len(X):
        raise ValueError('Need a 2D matrix and 2 <= n_clusters <= samples.')
    if epochs < 0 or pretrain_epochs < 0 or batch_size < 1:
        raise ValueError('Epoch counts must be nonnegative and batch_size positive.')
    if y is not None and len(y) != len(X):
        raise ValueError('y must align with X; labels are used only for reporting.')
    dimensions = [X.shape[1], 500, 500, 2000, 10] if dimensions is None else dimensions
    if model is None:
        activation = nn.Sigmoid() if loss == 'bce' else nn.Identity()
        model = VariationalAutoEncoder(dimensions, final_activation=activation)
    model = model.to(device)
    latent_dim = vae_encode(X[:min(2, len(X))], model).shape[1]
    mf = MatrixFactorizationBinaryModel(len(X), n_clusters, latent_dim).to(device)
    prehistory = {}
    if resume is None:
        prehistory = pre_train(X, model, pretrain_epochs, batch_size, lr, beta_end, loss,
                               seed, beta_start=beta_start, warm_epochs=warm_epochs)
        Z = vae_encode(X, model, batch_size)
        centers, _ = kmeans_plusplus(Z, n_clusters=n_clusters, random_state=seed)
        with torch.no_grad():
            mf.C.copy_(torch.as_tensor(centers, device=device))
    lr_C = lr if lr_C is None else lr_C
    lr_Y = lr if lr_Y is None else lr_Y
    optimizerC = torch.optim.Adam([{'params': [mf.C], 'lr': lr_C},
                                   {'params': model.parameters(), 'lr': lr}], lr=lr)
    optimizerY = torch.optim.Adam([{'params': [mf.Y], 'lr': lr_Y},
                                   {'params': model.parameters(), 'lr': lr}], lr=lr)
    schedulerC = torch.optim.lr_scheduler.StepLR(optimizerC, step_size=scheduler_step_size, gamma=scheduler_rate)
    schedulerY = torch.optim.lr_scheduler.StepLR(optimizerY, step_size=scheduler_step_size, gamma=scheduler_rate)
    history = {key: [] for key in ('loss', 'recon_loss', 'kl_loss', 'mf_loss',
                                   'beta', 'mf_weight', 'kappa', 'gamma')}
    start = 0
    settings = dict(n_clusters=n_clusters, batch_size=batch_size, lr=lr,
                    beta_start=beta_start, beta_end=beta_end, warm_epochs=warm_epochs,
                    mf_weight=mf_weight, kappa=kappa, gamma=gamma, binary_mode=binary_mode,
                    exclusive_mode=exclusive_mode, double_every=double_every, loss=loss, seed=seed,
                    model=repr(model), lr_Y=lr_Y, lr_C=lr_C, scheduler_step_size=scheduler_step_size,
                    scheduler_rate=scheduler_rate, mf_cycle=mf_cycle, prox_start_epoch=prox_start_epoch,
                    reinitialize_every=reinitialize_every, alphaX=alphaX, prox_lr=prox_lr, restart_kl=restart_kl)
    fingerprint = hashlib.sha256(X.numpy().tobytes()).hexdigest() if checkpoint or resume else None
    if resume is not None:
        state = torch.load(resume, map_location=device, weights_only=True)
        if state['fingerprint'] != fingerprint or state['settings'] != settings:
            raise ValueError('Resume needs the same samples, sample order, model, and training settings.')
        if epochs < state['epoch']:
            raise ValueError('epochs is the total target, and cannot be below the saved epoch.')
        model.load_state_dict(state['model'])
        mf.load_state_dict(state['mf'])
        optimizerC.load_state_dict(state['optimizerC'])
        optimizerY.load_state_dict(state['optimizerY'])
        schedulerC.load_state_dict(state['schedulerC'])
        schedulerY.load_state_dict(state['schedulerY'])
        start, history = state['epoch'], state['history']
        prehistory = state['prehistory']
        torch.set_rng_state(state['torch_rng'].cpu())
        if torch.cuda.is_available() and state['cuda_rng']:
            torch.cuda.set_rng_state_all([value.cpu() for value in state['cuda_rng']])
    for epoch in range(start, epochs):
        model.train()
        progress = min(epoch / max(warm_epochs, 1), 1.) if warm_epochs else 1.
        kl_epoch = epoch if restart_kl else len(prehistory.get('loss', [])) + epoch
        kl_progress = min(kl_epoch / max(warm_epochs, 1), 1.) if warm_epochs else 1.
        beta = beta_start + (beta_end - beta_start) * kl_progress
        weight = mf_weight * (.1 + .9 * progress)
        if mf_cycle:
            weight = mf_weight * min(2 * (epoch % mf_cycle) / mf_cycle, 1.)
        if reinitialize_every and epoch > 0 and epoch % reinitialize_every == 0:
            centers, _ = kmeans_fun_gpu(vae_encode(X, model, batch_size), K=n_clusters)
            with torch.no_grad():
                mf.C.copy_(torch.as_tensor(centers, device=device))
        multiplier = 2. ** (epoch // double_every) if double_every else 1.
        current_kappa, current_gamma = kappa * multiplier, gamma * multiplier
        totals, count = np.zeros(4), 0
        order = torch.randperm(len(X), generator=torch.Generator().manual_seed(seed + epoch))
        for cpu_idx in order.split(batch_size):
            idx, x = cpu_idx.to(device), X[cpu_idx].to(device)
            for block, optimizer in [('C', optimizerC), ('Y', optimizerY)]:
                rec, kl, mu = model_loss(model, x, loss)
                Y, C = mf.Y[idx], mf.C
                prediction = Y.detach() @ C if block == 'C' else Y @ C.detach()
                mf_loss = F.mse_loss(prediction, mu)
                value = rec + beta * kl + weight * mf_loss
                if block == 'Y':
                    previous_Y = mf.Y.detach().clone()
                optimizer_step(value, optimizer)
                if block == 'Y':
                    # Dense Adam has momentum on other rows; only this batch owns an update.
                    with torch.no_grad():
                        updated_rows = mf.Y[idx].clone()
                        mf.Y.copy_(previous_Y)
                        mf.Y[idx] = updated_rows
                    active = epoch >= prox_start_epoch
                    mf.apply_constraints(idx, current_kappa, current_gamma,
                                         binary_mode if active else 'none',
                                         exclusive_mode if active else 'none', bounded=True,
                                         alphaX=alphaX * multiplier, prox_lr=prox_lr)
                totals += np.array([value.item(), rec.item(), kl.item(), mf_loss.item()]) * len(x)
                count += len(x)
        schedulerC.step()
        schedulerY.step()
        row = dict(zip(('loss', 'recon_loss', 'kl_loss', 'mf_loss'), (totals / count).tolist()))
        row.update(beta=beta, mf_weight=weight, kappa=current_kappa, gamma=current_gamma)
        row.update(assignment_diagnostics(mf.Y))
        pred = mf.Y.detach().argmax(1).cpu().numpy()
        if y is not None:
            row.update(score_dict(y, pred))
        for key, value in row.items():
            history.setdefault(key, []).append(value)
        if verbose:
            metric = f', ACC={row["ACC"]:.4f}' if y is not None else ''
            print(f'{epoch + 1}/{epochs}: loss={row["loss"]:.4f}{metric}')
        if checkpoint:
            path = Path(checkpoint)
            path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(dict(model=model.state_dict(), mf=mf.state_dict(),
                            optimizerC=optimizerC.state_dict(), optimizerY=optimizerY.state_dict(),
                            schedulerC=schedulerC.state_dict(), schedulerY=schedulerY.state_dict(),
                            history=history, prehistory=prehistory, epoch=epoch + 1,
                            fingerprint=fingerprint, settings=settings, torch_rng=torch.get_rng_state(),
                            cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []), path)
    return dict(model=model, mf=mf, Z=vae_encode(X, model, batch_size),
                pred=mf.Y.detach().argmax(1).cpu().numpy(), history=history, prehistory=prehistory, pretrain_history=prehistory)

## MF on fixed features

This version receives pixels or an already computed representation. The encoder is not updated. With `binary_mode="none"`, Y is only constrained to be nonnegative and has no upper bound; C is signed, so this is Semi-NMF rather than fully nonnegative NMF. The NMF experiments are implemented separately in `matrix_factorization.ipynb`.

In [ ]:
def train_mf_binary_model(X, n_clusters=10, epochs=100, lr=1e-2, kappa=1e-4,
        gamma=0., binary_mode='improved', exclusive_mode='thesis', seed=42,
        Y_init=None, C_init=None, alphaX=1e-8, prox_lr=1e-4):
    seed_everything(seed)
    X = torch.as_tensor(X, dtype=torch.float32).cpu()
    if X.ndim != 2 or not 2 <= n_clusters <= len(X):
        raise ValueError('Need a 2D matrix and 2 <= n_clusters <= samples.')
    if C_init is None:
        C_init, _ = kmeans_plusplus(X.numpy(), n_clusters=n_clusters, random_state=seed)
    mf = MatrixFactorizationBinaryModel(len(X), n_clusters, X.shape[1], Y_init, C_init).to(device)
    optimizerC = torch.optim.Adam([mf.C], lr=lr)
    optimizerY = torch.optim.Adam([mf.Y], lr=lr)
    history = {'loss': [], 'mf_loss': []}
    for epoch in range(epochs):
        order = torch.randperm(len(X), generator=torch.Generator().manual_seed(seed + epoch))
        total = 0.
        for cpu_idx in order.split(256):
            idx, target = cpu_idx.to(device), X[cpu_idx].to(device)
            lossC = F.mse_loss(mf.Y[idx].detach() @ mf.C, target)
            optimizer_step(lossC, optimizerC)
            lossY = F.mse_loss(mf.Y[idx] @ mf.C.detach(), target)
            previous_Y = mf.Y.detach().clone()
            optimizer_step(lossY, optimizerY)
            with torch.no_grad():
                updated_rows = mf.Y[idx].clone()
                mf.Y.copy_(previous_Y)
                mf.Y[idx] = updated_rows
            mf.apply_constraints(idx, kappa, gamma, binary_mode, exclusive_mode,
                                 bounded=binary_mode != 'none', alphaX=alphaX, prox_lr=prox_lr)
            total += lossY.item() * len(idx)
        history['loss'].append(total / len(X))
        history['mf_loss'].append(total / len(X))
        for key, value in assignment_diagnostics(mf.Y).items():
            history.setdefault(key, []).append(value)
    return dict(mf=mf, Z=X.numpy(), pred=mf.Y.detach().argmax(1).cpu().numpy(), history=history)

## Training curves

In [ ]:
def plot_history(history, columns=('loss', 'recon_loss', 'kl_loss', 'mf_loss')):
    columns = [key for key in columns if key in history and len(history[key])]
    if not columns:
        return
    fig, axes = plt.subplots(1, len(columns), figsize=(4 * len(columns), 3), squeeze=False)
    for ax, key in zip(axes[0], columns):
        ax.plot(np.arange(1, len(history[key]) + 1), history[key])
        ax.set(xlabel='Epoch', ylabel=key)
        ax.grid(alpha=.2)
    fig.tight_layout()
    plt.show()